# Test existing plugins

## 1. Test doiorg plugin with one DOI

In [1]:
from doi_downloader import loader as ld, pdf_download as pdf_dl
import os

In [2]:
doi = "10.1038/s41586-025-10047-5"
correct_pdf_url = "https://www.nature.com/articles/s41586-025-10047-5.pdf"
outfile_name = "pdf_file.pdf"

Retrieve description file for DOI from https://doi.org and test if it is correct

In [3]:
pdf_url = ld.plugins["DoiorgPlugin"].get_pdf_url(doi, use_cache=False)
assert pdf_url == correct_pdf_url

Download PDF file from the url and check if the download was successful

In [4]:
try: os.remove(outfile_name)
except FileNotFoundError: pass
pdf_dl.download_pdf(pdf_url, outfile_name, ".", plugin_name="doiorg")
assert os.path.isfile(outfile_name), "Missing downloaded pdf file"
assert os.path.getsize(outfile_name) > 0, "Downloaded file is empty"

## 2. Test all plugins with multiple DOIs

Some of these plugins need an access key and will fail without such a key

### 2.1 Support functions

In [5]:
from doi_downloader import loader as ld, doi_downloader as ddl, pdf_download as pdf_dl
import json
import os
import polars as pl
import regex
from termcolor import colored
import time
import urllib

In [6]:
plugins = ld.plugins
PLUGINS = {"CoreacukPlugin": plugins['CoreacukPlugin'],
           "CrossrefPlugin": plugins['CrossrefPlugin'],
           "DoiorgPlugin": plugins['DoiorgPlugin'],
           "GoogleScholarSerpAPIPlugin": plugins['GoogleScholarSerpAPIPlugin'],
           "UnpaywallPlugin": plugins['UnpaywallPlugin']} # requires a commercial api key

CHAR_SUCCESS = "✅"
CHAR_FAILURE = "❌"

In [7]:
def make_safe_filename(doi):
    """Replace slashes and points in DOI by underscores, Add .pdf"""
    return doi.replace("/", "_").replace(".", "_") + ".pdf"

In [8]:
def get_pdf_urls_from_all_plugins(doi, debug=False):
    """Retrieve pdf_urls with all plugins"""
    urls = {}
    for plugin_name in PLUGINS.keys():
        try:
            plugin_data = PLUGINS[plugin_name].get_pdf_url(doi, use_cache=(plugin_name != "DoiorgPlugin"))
            try:
                plugin_data_json = json.loads(plugin_data)
                urls[plugin_name] = plugin_data_json["pdf_links"][0]
            except:
                urls[plugin_name] = plugin_data
            if debug: print("DEBUG:", plugin_name, urls[plugin_name], (plugin_name != "DoiorgPlugin"), urls)
        except Exception as e:
            print(f"   get_pdf_urls_from_all_plugins: error: {e}")
    return urls

In [9]:
def summarize_plugin_results(pdf_urls, index, result_type="url"):
    """Summarize plugin results with single icon: CHAR_SUCCESS or CHAR_FAILURE"""
    prefix =  f"   " if result_type == "url" else "   "
    print(f"{prefix} {result_type} retrieval success per plugin:", end=" ")
    for plugin_name in sorted(pdf_urls):
        print((CHAR_SUCCESS if pdf_urls[plugin_name] else CHAR_FAILURE), end=" ")
    print()

In [10]:
def download_pdf(doi, pdf_target_dir_name, pdf_target_file_name, pdf_urls):
    """Try to download pdf with each plugin, Register result"""
    target_locations = {}
    for plugin_name in pdf_urls:
        if pdf_urls[plugin_name]:
            target_locations[plugin_name] = pdf_dl.download_pdf(pdf_urls[plugin_name], 
                                                                pdf_target_file_name, 
                                                                pdf_target_dir_name,
                                                                plugin_name=plugin_name)
    for plugin_name in PLUGINS:
        if plugin_name not in target_locations or not target_locations[plugin_name]:
            target_locations[plugin_name] = None
    return target_locations

In [11]:
def check_plugin_access(doi_df):
    """Check plugin access to a list of DOIs: url retrieval and pdf downloads"""
    target_locations = []
    pdf_urls = []
    pdf_target_dir_name = "."
    for index, row in enumerate(doi_df.iter_rows()):
        doi = row[0]
        print(f"{index + 1}. {doi}")
        pdf_target_file_name = make_safe_filename(doi)
        pdf_urls.append(get_pdf_urls_from_all_plugins(doi, debug=False))
        summarize_plugin_results({key[0]: pdf_urls[-1][key[0]] 
                                  for key in sorted(pdf_urls[-1].items(), 
                                                    key=lambda item: item[0])}, 
                                 index + 1)
        target_locations.append(download_pdf(doi,
                                             pdf_target_dir_name, 
                                             pdf_target_file_name, 
                                             pdf_urls[-1]))
        summarize_plugin_results({key[0]: target_locations[-1][key[0]] 
                                          for key in sorted(target_locations[-1].items(), 
                                                     key=lambda item: item[0])}, 
                                          index + 1, result_type="pdf")
        if "MAX_PROCESSED" in globals() and index >= MAX_PROCESSED:
             break
    return pdf_urls, target_locations

In [12]:
def count_successes(pdf_urls):
    """Count the number of successes per plugin and their combination"""
    df_pdf_urls = pl.DataFrame(pdf_urls)
    df_pdf_urls = df_pdf_urls.null_count().select(sorted(df_pdf_urls.columns))
    pdf_url_never_found = len([True for row in pdf_urls if set(row.values()) == {None}])
    df_pdf_urls = df_pdf_urls.with_columns(pl.Series("Any", [pdf_url_never_found]))
    return df_pdf_urls.with_columns(pl.all() * -1 + len(pdf_urls))

### 2.2 doi_coda.csv (top 20)

The test DOIs were taken from the file `doi_coda.csv` (first twenty)

In [13]:
doi_coda_df = pl.DataFrame([{"doi": '10.1177/0022002714560349'},
                            {"doi": '10.1007/s10198-013-0496-x'},
                            {"doi": '10.1111/j.1465-7295.2010.00309.x'},
                            {"doi": '10.1016/j.econlet.2009.08.024'},
                            {"doi": '10.1038/s41467-017-00731-0'},
                            {"doi": '10.1093/ei/cbl001'},
                            {"doi": '10.1016/j.jesp.2004.09.004'},
                            {"doi": '10.1016/j.jpubeco.2014.01.005'},
                            {"doi": '10.1111/puar.12424'},
                            {"doi": '10.1016/j.geb.2006.09.005'},
                            {"doi": '10.1016/j.joep.2007.01.007'},
                            {"doi": '10.1108/03068290810854565'},
                            {"doi": '10.1111/j.1468-0335.2008.00689.x'},
                            {"doi": '10.1177/1069397108322455'},
                            {"doi": '10.1016/j.socec.2010.12.013'},
                            {"doi": '10.1017/s0022381609090355'},
                            {"doi": '10.1016/j.jpubeco.2008.06.007'},
                            {"doi": '10.1093/restud/rdt017'},
                            {"doi": '10.1016/j.socec.2011.08.028'},
                            {"doi": '10.1093/jae/ejr034'}])          

In [14]:
MAX_PROCESSED = 20

In [15]:
pdf_urls, target_locations = check_plugin_access(doi_coda_df[:MAX_PROCESSED])

1. 10.1177/0022002714560349
[coreacuk] Unauthorized access. Check your API key.
[crossref] using cached data for 10.1177/0022002714560349.
[doi.org] An error occurred: 403 Client Error: Forbidden for url: https://journals.sagepub.com/doi/10.1177/0022002714560349
[serpapi] using cached data for 10.1177/0022002714560349.
[unpaywall] using cached data for 10.1177/0022002714560349.
    url retrieval success per plugin: ❌ ✅ ❌ ✅ ❌ 
    pdf retrieval success per plugin: ❌ ❌ ❌ ❌ ❌ 
2. 10.1007/s10198-013-0496-x
[coreacuk] Unauthorized access. Check your API key.
[crossref] using cached data for 10.1007/s10198-013-0496-x.
[doi.org] robots.txt blocked acccess to https://link.springer.com/article/10.1007/s10198-013-0496-x
[serpapi] using cached data for 10.1007/s10198-013-0496-x.
[unpaywall] using cached data for 10.1007/s10198-013-0496-x.
    url retrieval success per plugin: ❌ ✅ ❌ ✅ ❌ 
    pdf retrieval success per plugin: ❌ ❌ ❌ ❌ ❌ 
3. 10.1111/j.1465-7295.2010.00309.x
[coreacuk] Unauthorized ac

In [16]:
count_successes(pdf_urls)

CoreacukPlugin,CrossrefPlugin,DoiorgPlugin,GoogleScholarSerpAPIPlugin,UnpaywallPlugin,Any
i64,i64,i64,i64,i64,i64
0,5,1,20,4,20


In [17]:
count_successes(target_locations)

CoreacukPlugin,CrossrefPlugin,DoiorgPlugin,GoogleScholarSerpAPIPlugin,UnpaywallPlugin,Any
i64,i64,i64,i64,i64,i64
0,1,1,5,3,6


### 2.3 browser plugin test set (14)

In [18]:
doi_plugin_df = pl.DataFrame([{"doi": "10.1613/jair.49"},
                              {"doi": "10.1038/s41586-025-10047-5"},
                              {"doi": "10.3390/electronics15040795"},
                              {"doi": "10.3389/fpsyt.2025.1739639"},
                              {"doi": "10.4236/jhrss.2026.141006"},
                              {"doi": "10.3897/aiep.51.63489"},
                              {"doi": "10.1016/j.nlp.2026.100202"},
                              {"doi": "10.1177/0022002714560349"},
                              {"doi": "10.1007/s10198-013-0496-x"},
                              {"doi": "10.1111/j.1465-7295.2010.00309.x"},
                              {"doi": "10.1016/j.econlet.2009.08.024"},
                              {"doi": "10.1093/ei/cb1001"},
                              {"doi": "10.2174/2213476X07666200423081738"},
                              {"doi": "10.1504/EJIM.2025.150039"}])

In [19]:
pdf_urls, target_locations = check_plugin_access(doi_plugin_df)

1. 10.1613/jair.49
[coreacuk] Unauthorized access. Check your API key.
[crossref] using cached data for 10.1613/jair.49.
[doi.org] webpage access problem for robots.txt: 403
[serpapi] using cached data for 10.1613/jair.49.
[unpaywall] using cached data for 10.1613/jair.49.
    url retrieval success per plugin: ❌ ✅ ✅ ✅ ✅ 
    pdf retrieval success per plugin: ❌ ✅ ✅ ✅ ✅ 
2. 10.1038/s41586-025-10047-5
[coreacuk] Unauthorized access. Check your API key.
[crossref] using cached data for 10.1038/s41586-025-10047-5.
[serpapi] using cached data for 10.1038/s41586-025-10047-5.
[unpaywall] using cached data for 10.1038/s41586-025-10047-5.
    url retrieval success per plugin: ❌ ✅ ✅ ✅ ❌ 
    pdf retrieval success per plugin: ❌ ✅ ✅ ✅ ❌ 
3. 10.3390/electronics15040795
[coreacuk] Unauthorized access. Check your API key.
[crossref] using cached data for 10.3390/electronics15040795.
[doi.org] An error occurred: 403 Client Error: Forbidden for url: https://www.mdpi.com/2079-9292/15/4/795
[serpapi] usin

In [20]:
count_successes(pdf_urls)

CoreacukPlugin,CrossrefPlugin,DoiorgPlugin,GoogleScholarSerpAPIPlugin,UnpaywallPlugin,Any
i64,i64,i64,i64,i64,i64
0,7,4,14,5,14


In [21]:
count_successes(target_locations)

CoreacukPlugin,CrossrefPlugin,DoiorgPlugin,GoogleScholarSerpAPIPlugin,UnpaywallPlugin,Any
i64,i64,i64,i64,i64,i64
0,3,4,6,3,7


## 3. In-depth testing of cause of pdf url retrieval failure with doi.org plugin

In [22]:
from doi_downloader.plugins.doiorg import DoiorgPlugin
import requests
from termcolor import colored
from time import sleep

In [23]:
URL_ACCESS_WAIT_TIME = 7

In [24]:
plugin = DoiorgPlugin()
cache = {}

def robot_access_allowed(url):
    if url not in cache:
        sleep(URL_ACCESS_WAIT_TIME)
        cache[url] = access_allowed = plugin.robot_access_allowed(url)
    return cache[url]

In [25]:
def color_code_status(status_code):
    if status_code == 200:
        return colored(status_code, "green")
    elif status_code >= 400:
        return colored(status_code, "red")
    else:
        return status_code

In [36]:
headers = { "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) "
                           "Chrome/120.0.0.0 Safari/537.36"),}

doi_df = doi_coda_df.clone()
status_code_counts = {}
for index, doi in enumerate(doi_df.get_columns()[0]):
    print(f"{index + 1:2d}. {doi}")
    sleep(URL_ACCESS_WAIT_TIME)
    response = requests.get("https://doi.org/" + doi, headers=headers, timeout=10)
    for r in response.history + [response]:
        print("   ", color_code_status(r.status_code), robot_access_allowed(r.url), r.url)
    if r.status_code not in status_code_counts:
        status_code_counts[r.status_code] = 0
    status_code_counts[r.status_code] += 1
status_code_counts

 1. 10.1177/0022002714560349
    302 True https://doi.org/10.1177/0022002714560349
    403 True https://journals.sagepub.com/doi/10.1177/0022002714560349
 2. 10.1007/s10198-013-0496-x
    302 True https://doi.org/10.1007/s10198-013-0496-x
    301 False http://link.springer.com/10.1007/s10198-013-0496-x
    303 False https://link.springer.com/article/10.1007/s10198-013-0496-x
    302 True https://idp.springer.com/authorize?response_type=cookie&client_id=springerlink&redirect_uri=https%3A%2F%2Flink.springer.com%2Farticle%2F10.1007%2Fs10198-013-0496-x
    200 False https://link.springer.com/article/10.1007/s10198-013-0496-x
 3. 10.1111/j.1465-7295.2010.00309.x
    302 True https://doi.org/10.1111/j.1465-7295.2010.00309.x
    403 True https://onlinelibrary.wiley.com/doi/10.1111/j.1465-7295.2010.00309.x
 4. 10.1016/j.econlet.2009.08.024
    302 True https://doi.org/10.1016/j.econlet.2009.08.024
    200 True https://linkinghub.elsevier.com/retrieve/pii/S0165176509002912
 5. 10.1038/s41467-01

{403: 10, 200: 10}

**Result coda DOIs**: ten of 20 DOIs result in a webpage retrieval with status code 200 (Success) while the other ten (1, 3, 6, 9, 12, 13, 14, 16, 18 and 20) produce a status code with value 403 (Access forbidden)<br>
**Result browser plugin DOIs**: eight of 14 DOIs result in a webpage retrieval with status code 200 (Success) while the other six (3, 5, 8, 10, 12 and 13) produce a status code with value 403 (Access forbidden)<br>
Results may vary because of unpredictable website behaviour

## 4. Playwright tests

In [27]:
import asyncio
from playwright.async_api import async_playwright

In [28]:
async def get_page(url):
    async with async_playwright() as p:
        browser = await p.firefox.launch()
        page = await browser.new_page()
        history = []
        page.on("response", lambda r: history.append((r.status, r.url)))
        response = await page.goto(url)
        content = await page.content()
        await browser.close()
        return response, content, history

In [38]:
doi_df = doi_coda_df.clone()
status_counts = {}
for index, doi in enumerate(list(doi_df.get_columns()[0])):
    print(f"{index + 1:2d}. {doi}")
    sleep(URL_ACCESS_WAIT_TIME)
    response, content, history = await get_page("https://doi.org/" + doi)
    print("   ", color_code_status(response.status), robot_access_allowed(response.url), response.url)
    if response.status not in status_counts:
        status_counts[response.status] = 0
    status_counts[response.status] += 1
status_counts

 1. 10.1177/0022002714560349
    403 True https://journals.sagepub.com/doi/10.1177/0022002714560349
 2. 10.1007/s10198-013-0496-x
    200 False https://link.springer.com/article/10.1007/s10198-013-0496-x
 3. 10.1111/j.1465-7295.2010.00309.x
    403 True https://onlinelibrary.wiley.com/doi/10.1111/j.1465-7295.2010.00309.x
 4. 10.1016/j.econlet.2009.08.024
    200 True https://linkinghub.elsevier.com/retrieve/pii/S0165176509002912
 5. 10.1038/s41467-017-00731-0
    200 True https://www.nature.com/articles/s41467-017-00731-0
 6. 10.1093/ei/cbl001
    403 True http://doi.wiley.com/10.1093/ei/cbl001
 7. 10.1016/j.jesp.2004.09.004
    200 True https://linkinghub.elsevier.com/retrieve/pii/S0022103104001180
 8. 10.1016/j.jpubeco.2014.01.005
    200 True https://linkinghub.elsevier.com/retrieve/pii/S0047272714000061
 9. 10.1111/puar.12424
    403 True https://onlinelibrary.wiley.com/doi/10.1111/puar.12424
10. 10.1016/j.geb.2006.09.005
    200 True https://linkinghub.elsevier.com/retrieve/pii/S0

{403: 7, 200: 13}

**Result coda DOIs** thirteen of 20 DOIs result in a webpage retrieval with status code 200 (Success) while the other seven (1, 3, 6, 9, 13, 14 and 16) produced a status code with value 403 (Access forbidden)<br>
**Result browser plugin DOIs** nine of 14 DOIs result in a webpage retrieval with status code 200 (Success) while four (8, 10, 12 and 14) produced a status code with value 403 (Access forbidden). One DOI (5) resulted in a time-out<br>
Results may vary because of unpredictable website behaviour

## 5. Summary

Success in PDF URL retrieval (columns 2-9) and PDF retrieval (columns 10-14) for the coda data set per DOI:

| id | coreacuk | crossref | doiorg | google | unpaywall | status code | playwright | difference | coreacuk | crossref | doiorg | google | unpaywall |
|:--:|:--------:|:--------:|:------:|:------:|:---------:|:-----------:|:----------:|:----------:|:--------:|:--------:|:------:|:------:|:---------:|
|  1 | ❌ | ✅ | ❌ | ✅ | ❌ | 403 | 403 | |
|  2 | ❌ | ✅ | ❌ | ✅ | ❌ | 200 | 200 | |
|  3 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 403 | | | | | ✅ | |
|  4 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | | | | | ✅ | |
|  5 | ❌ | ✅ | ✅ | ✅ | ✅ | 200 | 200 | | | ✅ | ✅ | ✅ | ✅ |
|  6 | ❌ | ❌ | ❌ | ✅ | ✅ | 403 | 403 | | | | | ✅ | ✅ |
|  7 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
|  8 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
|  9 | ❌ | ✅ | ❌ | ✅ | ✅ | 403 | 403 | |
| 10 | ❌ | ❌ | ❌ | ✅ | ✅ | 200 | 200 | | | | | | ✅ |
| 11 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
| 12 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 200 |!|
| 13 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 403 | |
| 14 | ❌ | ✅ | ❌ | ✅ | ❌ | 403 | 403 | |
| 15 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
| 16 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 403 | |
| 17 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
| 18 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 200 |!|
| 19 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
| 20 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 200 |!| | | | ✅ | |
|    |  0 |  5 |  1 | 20 |  4 |  10  | 13  |3|0|1|1|5|3| 

Success in PDF URL retrieval (columns 2-9) and PDF retrieval (columns 10-14) for the browser plugin data set per DOI:

| id | coreacuk | crossref | doiorg | google | unpaywall | status code | playwright | difference | coreacuk | crossref | doiorg | google | unpaywall |
|:--:|:--------:|:--------:|:------:|:------:|:---------:|:-----------:|:----------:|:----------:|:--------:|:--------:|:------:|:------:|:---------:|
|  1 | ❌ | ✅ | ✅ | ✅ | ✅ | 200 | 200 | | | ✅ | ✅ | ✅ | ✅ |
|  2 | ❌ | ✅ | ✅ | ✅ | ❌ | 200 | 200 | | | ✅ | ✅ | ✅ | |
|  3 | ❌ | ❌ | ❌ | ✅ | ✅ | 403 | 200 |!|
|  4 | ❌ | ❌ | ✅ | ✅ | ✅ | 200 | 200 | | | | ✅ | | ✅|
|  5 | ❌ | ✅ | ❌ | ✅ | ✅ | 403 | 200 |!| | | | | |
|  6 | ❌ | ✅ | ✅ | ✅ | ❌ | 200 | 200 | | | ✅ | ✅ | ✅ | |
|  7 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | |
|  8 | ❌ | ✅ | ❌ | ✅ | ❌ | 403 | 403 | |
|  9 | ❌ | ✅ | ❌ | ✅ | ❌ | 200 | 200 | |
| 10 | ❌ | ❌ | ❌ | ✅ | ❌ | 403 | 403 | | | | | ✅ | |
| 11 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 200 | | | | | ✅ | |
| 12 | ❌ | ❌ | ❌ | ✅ | ✅ | 403 | 403 |!| | | | ✅ | ✅ |
| 13 | ❌ | ✅ | ❌ | ✅ | ❌ | 403 | 200 | |
| 14 | ❌ | ❌ | ❌ | ✅ | ❌ | 200 | 403 |?|
|    |  0 |  7 |  4 | 14 |  5 |   6  |  4  |4|0|3|4|6|4|
